# 1. Planificación Básica de Agentes

## Objetivos de Aprendizaje
- Comprender qué es la planificación en el contexto de los agentes de IA.
- Diferenciar entre un agente reactivo (como los vistos hasta ahora) y un agente planificador.
- Implementar un agente básico de tipo **Plan-and-Execute**.
- Entender las limitaciones de la planificación estática y la necesidad de la orquestación.

## De la Reacción a la Planificación

En los módulos anteriores, hemos construido agentes que son fundamentalmente **reactivos**. El ciclo ReAct (Reason + Act) es potente, pero solo decide la siguiente mejor acción a tomar. Es como un trabajador que, después de cada tarea, vuelve a preguntar: '¿Y ahora qué hago?'

Esto funciona para tareas simples, pero falla ante objetivos complejos que requieren múltiples pasos interdependientes. Por ejemplo, si le pedimos a un agente: 'Investiga sobre la empresa NVIDIA, encuentra el nombre de su CEO y luego busca la biografía de esa persona en Wikipedia', un agente puramente reactivo podría perderse.

La **planificación** es la capacidad de un agente para descomponer un objetivo complejo en una secuencia de pasos más pequeños y ejecutables **antes** de empezar a actuar. Es la diferencia entre reaccionar al momento y tener una estrategia.

### Tipos de Agentes Planificadores

Existen varias arquitecturas para la planificación, pero dos de las más comunes son:

1.  **Agentes Plan-and-Execute**: 
    - **Cómo funcionan**: Primero, llaman a un LLM para crear un plan completo, paso a paso. Luego, ejecutan esos pasos en orden, sin desviarse del plan original.
    - **Ventajas**: Simple de implementar y predecible.
    - **Desventajas**: El plan es estático. Si un paso falla o la información obtenida en un paso intermedio invalida el resto del plan, el agente no puede adaptarse.

2.  **Agentes ReAct (y similares)**:
    - **Cómo funcionan**: Aunque los hemos llamado reactivos, también son una forma de planificación incremental. Planean y ejecutan un paso a la vez, y el resultado de cada acción influye en la planificación del siguiente paso.
    - **Ventajas**: Son dinámicos y pueden corregir el rumbo sobre la marcha.
    - **Desventajas**: Pueden entrar en bucles o perder de vista el objetivo general si se desvían demasiado.

En este notebook, construiremos un agente **Plan-and-Execute** básico para ilustrar el concepto de forma clara.

!pip install langchain-openai openai wikipedia -q

In [2]:
!pip install langchain-openai openai wikipedia -q

In [3]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        base_url=os.environ.get("GITHUB_BASE_URL"),
        api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. El Planificador: Creando el Plan

Nuestro primer paso es usar el LLM no para ejecutar una tarea, sino para **crear un plan**. Le daremos un objetivo y le pediremos que nos devuelva una lista de pasos claros y concisos.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un excelente planificador de tareas. Tu trabajo es tomar un objetivo complejo y descomponerlo en una lista de pasos simples y ejecutables. Responde únicamente con la lista de pasos. Utiliza unicamente la wikipedia para las fuentes"),
    ("human", "Objetivo: {goal}")
])

planner_chain = planner_prompt | llm

goal = "Investiga sobre la empresa Tesla, encuentra el nombre de su CEO y luego busca la biografía de esa persona en Wikipedia."

plan = planner_chain.invoke({"goal": goal})

print("--- PLAN GENERADO ---")
print(plan.content)

--- PLAN GENERADO ---
1. Abre Wikipedia en tu navegador.  
2. Busca "Tesla, Inc." en Wikipedia.  
3. Lee el artículo para identificar el nombre del CEO actual de Tesla.  
4. Copia el nombre del CEO identificado.  
5. Realiza una nueva búsqueda en Wikipedia utilizando el nombre del CEO.  
6. Accede al artículo de Wikipedia sobre el CEO.  
7. Lee la biografía del CEO en el artículo encontrado.  


### 3. El Ejecutor: Siguiendo el Plan

Ahora que tenemos un plan en formato de texto, necesitamos un **ejecutor**. El ejecutor leerá cada paso del plan y utilizará una herramienta (en este caso, una simple búsqueda en Wikipedia) para completarlo. 

Para este ejemplo básico, nuestro ejecutor será una simple función de Python que interpreta cada línea del plan.

In [8]:
def execute_plan(plan_text):
    steps = plan_text.strip().split('\n')
    results = []
    
    print("--- EJECUTANDO PLAN ---")
    for i, step in enumerate(steps):
        # Extraer la acción y el sujeto de cada paso (esto es muy básico, en un agente real se usarían herramientas más robustas)
        clean_step = step.split('.', 1)[-1].strip() # Quita el número del paso
        print(f"[Paso {i+1}] Ejecutando: {clean_step}")
        
        try:
            # Asumimos que cada paso es una búsqueda en Wikipedia
            # En un agente real, aquí se seleccionaría la herramienta adecuada
            query = clean_step.split('sobre ')[-1].split(' de ')[-1] # Intenta adivinar el sujeto de la búsqueda
            result = wikipedia.summary(query, sentences=2)
            print(f"Resultado: {result}")
            results.append(result)
        except Exception as e:
            error_message = f"No se pudo completar el paso: {e}"
            print(error_message)
            results.append(error_message)
            
    return results

# Ejecutamos el plan generado por el LLM
execution_results = execute_plan(plan.content)

--- EJECUTANDO PLAN ---
[Paso 1] Ejecutando: Abre Wikipedia en tu navegador.
No se pudo completar el paso: Page id "abreu wikipedia en tu navegadores" does not match any pages. Try another id!
[Paso 2] Ejecutando: Busca "Tesla, Inc." en Wikipedia.
Resultado: Kimbal Musk (Pretoria, Sudáfrica; 20 de septiembre de 1972) es un empresario de restaurantes y filántropo canado-sudafricano nacionalizado estadounidense. En 1995 cofundó con su hermano Elon Musk la empresa de software Zip2, que en 1999 fue comprada por Compaq por un valor de 307 millones de USD.​​
En los últimos años ha creado varias empresas con el objetivo de tener un impacto duradero en la cultura de la comida mediante restaurantes, huertos escolares y granjas urbanas.​
Kimbal Musk y Hugo Matheson fundaron en 2004 The Kitchen Restaurant Group, un grupo de restaurantes de proximidad en Colorado, Chicago, Cleveland,​ Memphis,​ e Indianapolis.​ Desde 2011 es cofundador y presidente de Big Green (anteriormente The Kitchen Community

Resultado: En la mitología griega, Ceo (en griego: Κοῖος, Koíos; latín: Coeus; «inteligente»)​  era uno de los doce titanes, hijos de Urano (Cielo) y Gea (Tierra).​  Como los demás titanes, Ceo fue derrocado por Zeus tras la Titanomaquia, que lo encerró en el Tártaro, y desde entonces nada más se ha vuelto a saber de él.​  Sus hijas llevaban el patronímico de Ceides​ o Ceántides.​  Ceo es un personaje oscuro que apenas aparece en los textos mitográficos, e incluso Ovidio alegaba que «Leto era hija Ceo, quienquiera que fuera este».​  Las versiones romanas lo incluyen entre los gigantes​ e incluso lo denominan como Polo (Polus, en relación con el polo).​


== Etimología ==
Cornuto, jugando con su etimología, dice que Ceo era una suerte de personificación de la percepción (koeîn), esto es, la acción de «entender» o «pensar», y de ahí su relación como titán de la inteligencia.​ 


== En la Teogonía ==
En la Teogonía se nos habla de su matrimonio y descendencia, a saber: «a su vez Febe visi

### 4. Conclusiones y Transición a la Orquestación

Hemos creado con éxito un agente que puede planificar y luego ejecutar. Este es un gran paso adelante con respecto a un agente puramente reactivo.

Sin embargo, las **limitaciones** de nuestro enfoque simple son evidentes:

1.  **Plan Estático**: El plan no puede cambiar. Si el CEO de Tesla no fuera Elon Musk, el plan seguiría buscando la biografía de Elon Musk.
2.  **Ejecutor Frágil**: Nuestro ejecutor es muy simple. Asume que cada paso es una búsqueda en Wikipedia y extrae el tema de búsqueda de forma muy rudimentaria.
3.  **Falta de Contexto entre Pasos**: El resultado del paso 1 (investigar sobre Tesla) no se utiliza en el paso 2. El agente no "sabe" que el CEO que encontró en el primer paso es la persona que debe buscar en el segundo.

Estos problemas son precisamente los que resuelve la **orquestación de agentes**. Un orquestador es un sistema más inteligente que no solo crea un plan, sino que también:
- Gestiona el flujo de datos entre los pasos.
- Selecciona la herramienta correcta para cada tarea.
- Puede manejar errores y adaptar el plan si es necesario.
- Coordina la colaboración entre diferentes agentes especializados.

En los próximos notebooks, exploraremos cómo frameworks como **LangChain y CrewAI** nos proporcionan herramientas de orquestación para construir agentes mucho más potentes y robustos, superando las limitaciones de nuestro planificador básico.